# Feature Stores & Data Pipelines (Deep Dive) — AI Lab Instructor Notebook

*System Design · Expert*



6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import json
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

def pretty(d):
    print(json.dumps(d, indent=2, sort_keys=True))

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Feature Stores & Data Pipelines (Deep Dive) — Student Lab

Offline lab: build point-in-time correct offline features, contrast them with online materialized features, and show how training-serving skew appears.

Run the tasks in order. Later sections depend on the event stream and feature tables defined earlier.

In [ ]:
pretty({'lab': 'feature stores deep dive', 'focus': ['PIT correctness', 'online materialization', 'parity / skew', 'metadata']})

**Why this works.** This lab is about the contract between historical training features and low-latency online features. Keep that distinction in mind for every section.

## Task 2: 0 — Generate synthetic event stream

## 0 — Generate synthetic event stream

Create a timestamped event stream with page views, add-to-carts, and purchases. Sort the events by time because later feature logic assumes temporal order.

In [ ]:
$f

**Why this works.** The event stream is the source of truth. Sorting by time is essential because point-in-time features depend on temporal ordering.

## Task 3: 1 — Offline features (point-in-time correct)

## 1 — Offline features (point-in-time correct)

For each historical training row, compute user features from only the prior 24 hours of activity. Do not use events at or after the prediction timestamp.

In [ ]:
$10

**Why this works.** This section is the core of the lab. The same user gets different feature values at different prediction times, so the join must be relative to each row, not just the latest state.

## Task 4: 2 — Online feature store (materialization)

## 2 — Online feature store (materialization)

Build the latest-value online feature table as of end-of-stream. This is the right shape for serving, but not automatically safe for offline training.

In [ ]:
$11

**Why this works.** The online store materializes the latest keyed values for low-latency lookup. It is intentionally not point-in-time and can leak future information if reused carelessly.

## Task 5: 3 — Offline/online parity check (skew demo)

## 3 — Offline/online parity check (skew demo)

Compare the online latest totals against the PIT-correct historical features for the same users. This demonstrates why naive offline joins often overestimate feature values.

In [ ]:
naive_pv = feat_on['pv_total'][user_ids]
pit_pv = feat_off['pv_24h']
naive_spend = feat_on['spend_total'][user_ids]
pit_spend = feat_off['spend_24h']

print('naive_pv_mean', float(naive_pv.mean()), 'pit_pv_mean', float(pit_pv.mean()))
print('naive_spend_mean', float(naive_spend.mean()), 'pit_spend_mean', float(pit_spend.mean()))
check('naive_ge_pit_often', float((naive_pv >= pit_pv).mean()) > 0.70)
check('spend_skew_visible', float((naive_spend >= pit_spend).mean()) > 0.70)

**Why this works.** This skew demo is the practical warning sign: the online latest-value table usually dominates the historical PIT values because it contains future events.

## Task 6: 4 — Data quality + lineage metadata

## 4 — Data quality + lineage metadata

Attach ownership and freshness metadata to the offline features and run a few basic validation checks. The goal is to treat feature tables as production artifacts, not just arrays.

In [ ]:
$12

**Why this works.** Production features need metadata and validation. Ownership, freshness tier, and PIT-safety are part of the feature contract, not optional documentation.